# Chapter 10 — SQLAlchemy and Pandas Database Workflows

SQLAlchemy provides database connectivity and SQL expression/ORM layers. Pandas bridges SQL queries and DataFrames. Learn engines, connections, transactions, bind parameters, and table exchange.

## Prerequisites
Chapter 9 SQL. Install SQLAlchemy; MySQL driver is optional for the SQLite demonstrations.

## Learning goals
- Distinguish Engine, Connection, Session, and ORM concepts.
- Create database URLs and configure connection behavior.
- Execute bound SQL with text().
- Read and write DataFrames to relational databases safely.

**Study sequence:** use the theory notebook first, then tutorial, then lab. The examples use the bundled synthetic data unless otherwise noted.

In [1]:
from pathlib import Path
ROOT=Path.cwd()
if not (ROOT/"data").exists() and (ROOT.parent/"data").exists(): ROOT=ROOT.parent
DATA=ROOT/"data"
print("Data directory:", DATA.resolve())

Data directory: /mnt/data/python_data_analysis_book/data


## Guided learning path

Work from top to bottom. Each step states the purpose, explains the API choice, and provides a complete code cell. Modify parameters and predict the output before running it.

### Step 1 — Guided 1 — Use transactions

The `engine.begin()` context commits on success and rolls back on an exception.

In [2]:
from sqlalchemy import create_engine,text
engine=create_engine("sqlite:///:memory:")
with engine.begin() as con: con.execute(text("CREATE TABLE t(id INTEGER PRIMARY KEY,value TEXT)")); con.execute(text("INSERT INTO t VALUES(1,'ready')"))
print("table created")

table created


### Step 2 — Guided 2 — Read with Pandas

Use a checked-out Connection for `pd.read_sql`; parameter values should be bound separately.

In [3]:
import pandas as pd
with engine.connect() as con: frame=pd.read_sql(text("SELECT * FROM t WHERE id=:id"),con,params={"id":1})
display(frame)

   id  value
0   1  ready


### Step 3 — Guided 3 — Append a frame

`if_exists="append"` adds rows; `replace` drops/recreates the table and is not a safe general production migration strategy.

In [4]:
new=pd.DataFrame([{"id":2,"value":"review"}]); new.to_sql("t",engine,if_exists="append",index=False)
with engine.connect() as con: display(pd.read_sql(text("SELECT * FROM t ORDER BY id"),con))

   id   value
0   1   ready
1   2  review


### Step 4 — Guided 4 — Configure MySQL

The URL object keeps passwords out of manual string interpolation and can properly handle special characters.

In [5]:
from sqlalchemy.engine import URL
example=URL.create("mysql+pymysql",username="analyst",password="example-only",host="localhost",database="analytics")
print(example.drivername,example.host,example.database)

mysql+pymysql localhost analytics


## Tutorial checkpoint

1. What was the input data type?
2. What did the function or method return?
3. Which argument most changed the result?
4. What edge case should be tested before using this in production?